In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Read the CSV file
df = pd.read_csv('1k_fit_parameters.csv')

# Get unique values for selectors
particles = sorted(df['PID'].unique())
etas = sorted(df['pseudorapidity'].unique())

# Convert eta to string for better display
df['eta_label'] = df['pseudorapidity'].round(1).astype(str)

# Create the figure with subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('ca_thtrA_mean vs Momentum', 'cg_thtrG_mean vs Momentum'),
    shared_xaxes=True,
    horizontal_spacing=0.15
)

# Add initial traces (empty or with all data)
for particle in particles:
    particle_data = df[df['PID'] == particle]
    
    fig.add_trace(
        go.Scatter(
            x=particle_data['momentum'],
            y=particle_data['ca_thtrA_mean'],
            mode='markers',
            name=f'PID {particle}',
            legendgroup=f'pid_{particle}',
            marker=dict(size=8),
            hovertemplate='<b>PID: %{text}</b><br>' +
                         'Momentum: %{x:.1f}<br>' +
                         'ca_mean: %{y:.3f} ± %{customdata[0]:.3f}<br>' +
                         'η: %{customdata[1]:.1f}<extra></extra>',
            text=particle_data['PID'],
            customdata=np.column_stack([particle_data['ca_thtrA_mean_err'], 
                                       particle_data['eta']]),
            visible=True
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=particle_data['momentum'],
            y=particle_data['cg_thtrG_mean'],
            mode='markers',
            name=f'PID {particle}',
            legendgroup=f'pid_{particle}',
            marker=dict(size=8),
            hovertemplate='<b>PID: %{text}</b><br>' +
                         'Momentum: %{x:.1f}<br>' +
                         'cg_mean: %{y:.3f} ± %{customdata[0]:.3f}<br>' +
                         'η: %{customdata[1]:.1f}<extra></extra>',
            text=particle_data['PID'],
            customdata=np.column_stack([particle_data['cg_thtrG_mean_err'], 
                                       particle_data['pseudorapidity']]),
            visible=True,
            showlegend=False  # Hide duplicate legend entries
        ),
        row=1, col=2
    )

# Create buttons for particle selection
pid_buttons = []
all_pids_visible = [True] * (len(particles) * 2)  # Twice as many traces (2 subplots)

# Button for all particles
pid_buttons.append(
    dict(
        label='All Particles',
        method='update',
        args=[{'visible': all_pids_visible},
              {'title': 'All Particles'}]
    )
)

# Button for each particle
for i, particle in enumerate(particles):
    visible = [False] * (len(particles) * 2)
    visible[i] = True          # Show in first subplot
    visible[i + len(particles)] = True  # Show in second subplot
    pid_buttons.append(
        dict(
            label=f'PID {particle}',
            method='update',
            args=[{'visible': visible},
                  {'title': f'Particle {particle}'}]
        )
    )

# Create buttons for eta selection
eta_buttons = []
eta_values = sorted(df['pseudorapidity'].unique())

# Button for all eta
all_eta_visible = [True] * (len(particles) * 2)
eta_buttons.append(
    dict(
        label='All η',
        method='update',
        args=[{'visible': all_eta_visible}]
    )
)

# Button for each eta
for eta in eta_values:
    eta_data = df[df['pseudorapidity'] == eta]
    pids_in_eta = eta_data['PID'].unique()
    
    visible = [False] * (len(particles) * 2)
    for pid in pids_in_eta:
        idx = list(particles).index(pid)
        visible[idx] = True
        visible[idx + len(particles)] = True
    
    eta_buttons.append(
        dict(
            label=f'η = {eta:.1f}',
            method='update',
            args=[{'visible': visible}]
        )
    )

# Update layout with buttons
fig.update_layout(
    title='Mean Values vs Momentum by Particle and Pseudorapidity',
    xaxis_title='Momentum',
    yaxis_title='ca_thtrA_mean',
    xaxis2_title='Momentum',
    yaxis2_title='cg_thtrG_mean',
    hovermode='closest',
    updatemenus=[
        # Particle selection dropdown
        dict(
            buttons=pid_buttons,
            direction='down',
            showactive=True,
            x=0.1,
            xanchor='left',
            y=1.15,
            yanchor='top'
        ),
        # Eta selection dropdown
        dict(
            buttons=eta_buttons,
            direction='down',
            showactive=True,
            x=0.4,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    ],
    annotations=[
        dict(text='Select Particle:', x=0, xref='paper', y=1.1, yref='paper',
             align='left', showarrow=False),
        dict(text='Select η:', x=0.32, xref='paper', y=1.1, yref='paper',
             align='left', showarrow=False)
    ]
)

# Add error bars (optional - uncomment if you want error bars)
# for i in range(len(particles)):
#     fig.add_trace(
#         go.Scatter(
#             x=df[df['PID'] == particles[i]]['momentum'],
#             y=df[df['PID'] == particles[i]]['ca_thtrA_mean'],
#             error_y=dict(
#                 type='data',
#                 array=df[df['PID'] == particles[i]]['ca_thtrA_mean_err'],
#                 visible=True
#             ),
#             showlegend=False,
#             mode='markers',
#             marker=dict(color='rgba(0,0,0,0)')
#         ),
#         row=1, col=1
#     )

fig.show()

# Optional: Save as HTML file
# fig.write_html('mean_values_interactive.html')